# Hotspot_Detection.ipynb — Geospatial Accident Hotspot Clustering

DBSCAN on lat/lon, weighted by severity, run per city. Uses `cleaned_accidents.csv`
(no `accident_id` column, so row counts use `.size()` / index length instead).

## 1. Imports & Load

In [23]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import folium
from folium.plugins import HeatMap
df = pd.read_csv('../data/cleaned_accidents.csv')
df.head()

,city,state,latitude,longitude,hour,day_of_week,is_weekend,road_type,lanes,traffic_signal,weather,visibility,temperature,traffic_density,cause,accident_severity,vehicles_involved,casualties,is_peak_hour,risk_score
0,Pune,Maharashtra,18.680827,73.930388,5,Sunday,1,highway,3,1,fog,low,32,high,weather,fatal,2,2,0,0.85
1,Mumbai,Maharashtra,18.817732,72.790846,4,Sunday,1,urban,4,0,clear,high,34,low,weather,major,4,3,0,0.10
2,Mumbai,Maharashtra,19.096889,72.819424,13,Wednesday,0,urban,3,0,fog,low,21,medium,weather,minor,1,1,0,0.45
3,Chandigarh,Punjab,30.787805,76.847507,11,Sunday,1,urban,1,1,fog,low,30,high,distraction,minor,5,2,0,0.65
4,Chennai,Tamil Nadu,12.965155,80.283313,16,Thursday,0,highway,3,1,clear,high,24,low,distraction,minor,2,1,0,0.10


In [24]:
from pathlib import Path

# Resolve data/ folder regardless of whether the kernel's cwd is the
# project root (GEOSAFE/) or the notebooks/ folder itself.
DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('../data')
print('Using DATA_DIR:', DATA_DIR.resolve())

Using DATA_DIR: C:\Users\Pritam\Desktop\GeoSafe\data


## 2. Severity Weighting

In [25]:
severity_weight = {'minor': 1, 'major': 2, 'fatal': 3}
df['severity_weight'] = df['accident_severity'].map(severity_weight)

## 3. DBSCAN Clustering (per city)

`eps_km=0.5` -> 500m radius; `min_samples=5` -> at least 5 accidents within that
radius to count as a hotspot. `cluster == -1` is noise (non-hotspot).

In [26]:
def cluster_city(city_df, eps_km=0.5, min_samples=5):
    coords = city_df[['latitude','longitude']].to_numpy()
    coords_rad = np.radians(coords)
    eps_rad = eps_km / 6371.0  # earth radius in km
    db = DBSCAN(eps=eps_rad, min_samples=min_samples, metric='haversine').fit(coords_rad)
    city_df = city_df.copy()
    city_df['cluster'] = db.labels_
    return city_df

clustered = df.groupby('city', group_keys=False).apply(cluster_city)
print(clustered['cluster'].value_counts().head())

cluster
-1     14843
 2       315
 18      213
 10      174
 14      165
Name: count, dtype: int64


C:\Users\Pritam\AppData\Local\Temp\ipykernel_7788\3498570732.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  clustered = df.groupby('city', group_keys=False).apply(cluster_city)


## 4. Rank Hotspots by Weighted Risk

`accident_count` now uses `.size()` since there's no `accident_id` to count.

In [27]:
hotspots = clustered[clustered['cluster'] != -1].groupby(['city','cluster']).agg(
    lat=('latitude','mean'),
    lon=('longitude','mean'),
    accident_count=('cluster','size'),
    avg_risk_score=('risk_score','mean'),
    total_severity_weight=('severity_weight','sum')
).reset_index().sort_values('total_severity_weight', ascending=False)

hotspots.head(15)

,city,cluster,lat,lon,accident_count,avg_risk_score,total_severity_weight
27,Chandigarh,2,30.619949,76.801194,261,0.461111,418
43,Chandigarh,18,30.740439,76.833232,167,0.428144,285
35,Chandigarh,10,30.664220,76.794446,135,0.443333,213
39,Chandigarh,14,30.665427,76.731810,131,0.429389,211
36,Chandigarh,11,30.672556,76.867274,111,0.447748,185
34,Chandigarh,9,30.655652,76.832801,96,0.489583,152
42,Chandigarh,17,30.754769,76.884917,100,0.447500,149
44,Chandigarh,19,30.614382,76.716206,80,0.433125,133
48,Chandigarh,23,30.720669,76.882986,63,0.419048,99
53,Chandigarh,28,30.789430,76.839110,54,0.486111,87


## 5. Visualize on Interactive Map

In [29]:
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5)

heat_data = df[['latitude','longitude','severity_weight']].values.tolist()
HeatMap(heat_data, radius=12, blur=15).add_to(m)

for _, row in hotspots.head(15).iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=8,
        color='red',
        fill=True,
        popup=f"{row['city']} | Accidents: {row['accident_count']} | Risk: {row['avg_risk_score']:.2f}"
    ).add_to(m)

m.save(DATA_DIR / 'hotspot_map.html')


## 6. Export Hotspots for the Streamlit App

Saves a clean `hotspots.csv` with a `risk_level` (High/Medium/Low) tier so the app doesn't need to rerun DBSCAN every time it loads.

In [30]:
# Export for the Streamlit app (avoids recomputing DBSCAN on every app run)
hotspots_export = hotspots.copy()

# Risk tiering by weighted severity (quantile-based: top 25% = High, next 35% = Medium, rest = Low)
q_high = hotspots_export['total_severity_weight'].quantile(0.75)
q_med = hotspots_export['total_severity_weight'].quantile(0.40)

def tier(w):
    if w >= q_high:
        return 'High'
    elif w >= q_med:
        return 'Medium'
    return 'Low'

hotspots_export['risk_level'] = hotspots_export['total_severity_weight'].apply(tier)
hotspots_export.to_csv(DATA_DIR / 'hotspots.csv', index=False)

print(hotspots_export['risk_level'].value_counts())
print("Saved:", (DATA_DIR / 'hotspots.csv').resolve())

risk_level
Medium    224
Low       140
High      127
Name: count, dtype: int64
Saved: C:\Users\Pritam\Desktop\GeoSafe\data\hotspots.csv
